# time-stage-instrumentation — worked example 3: Exception-safe stage timer

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `time-stage-instrumentation`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Wrapping the timer in a `@contextmanager` with a `try/finally` makes instrumentation crash-safe: even if the timed block raises, the `finally` still records the elapsed time before the exception propagates. The accumulator uses `dict.get(name, 0.0)` so a fresh dict works without pre-seeding keys.

## Worked solution

We build a reusable `stage(name, acc)` context manager and use it to time two blocks.

1. The function is decorated with `@contextlib.contextmanager`; it records `t0` at entry, `yield`s, and in a `finally` clause adds the elapsed seconds into `acc[name]`.
2. Using `acc.get(name, 0.0) + elapsed` means the first time a name is seen it starts from zero — no need to initialize keys ahead of time.
3. We demonstrate it timing a 'load' block and a 'train' block; both totals accumulate into the same dict.
4. The `finally` guarantees the time is recorded even on an exception, which is why instrumentation built this way never loses a measurement.

The printed dict shows both stages recorded with positive durations.

In [ ]:
import time
import contextlib

@contextlib.contextmanager
def stage(name, acc):
    t0 = time.perf_counter()
    try:
        yield
    finally:
        acc[name] = acc.get(name, 0.0) + (time.perf_counter() - t0)

acc = {}
for _ in range(2):
    with stage('load', acc):
        time.sleep(0.003)
    with stage('train', acc):
        time.sleep(0.006)
print('load recorded:', acc['load'] > 0)
print('train recorded:', acc['train'] > 0)
print('train slower than load:', acc['train'] > acc['load'])